In [ ]:
%pip install -r requirements.txt  # install runtime dependencies when running the notebook

In [ ]:
import pandas as pd
data=pd.read_excel("fplot_sample_data.xlsx")

In [ ]:
data.columns

In [ ]:
import forestplot as fp
from matplotlib import font_manager as fm, rcParams
import re, os
# Choose a display font name (e.g. 'Arial'). If available locally, it will be registered and used;
# otherwise a sensible fallback will be selected so rendering succeeds.
chosen_font = 'Arial'  # change this to any font name you want to test
def register_font_by_name(name):
    # search both basename and full path for the name (case-insensitive)
    candidates = [p for p in fm.findSystemFonts() if name.lower() in os.path.basename(p).lower() or name.lower() in p.lower()]
    if candidates:
        fm.fontManager.addfont(candidates[0])
        rcParams['font.family'] = fm.FontProperties(fname=candidates[0]).get_name()
        print('Using registered font', rcParams['font.family'], 'from', candidates[0])
        return True
    return False
# Try the user's choice, then fallbacks if not available
if not register_font_by_name(chosen_font):
    for alt in ['DejaVu Sans','Noto Sans','Liberation Sans','DejaVuSans','DejaVu Sans Mono']:
        if register_font_by_name(alt):
            break
    else:
        print('No matching fonts found for chosen or fallbacks; using existing default', rcParams.get('font.family'))
# Draw the forest plot (this uses rcParams['font.family'] for labels)
ax = fp.forestplot(
    data.round(4), # the dataframe with results data
    estimate="Hazard ratio", ll="HR lower 95%", hl="HR upper 95%",
    varlabel="label", capitalize="capitalize", pval="p-value",
    annote=["est_ci"], annoteheaders=["Hazard ratio (95% CI)"],
    rightannote=["p-value"], right_annoteheaders=["P-value"],
    groupvar="group",
    group_order=['Age','Male sex','Hispanic','Race','Insurance status','Charlson-Deyo Score'],
    xlabel="Hazard ratios", **{"xlabel_size":15}, xlim=(0.5,1.5), table=True,
    **{"marker":"D","xline":1.0,"markersize":20,"xlinestyle":(0,(10,5)),"xlinecolor":"#FF0000",
       "xtick_size":12,"fontsize":15,"figsize":(5,10)}
)
# Post-process annotation/table text: force the numeric/CI column to use a monospace font so alignment is preserved
mono_fp = fm.FontProperties(fname='fonts/DejaVuSansMono.ttf')
num_regex = re.compile(r'^[\d\s\.,\(\)\-–—/%+eE:]+$')  # numbers, punctuation, parentheses, dashes, percent, exponent
for txt in ax.texts:
    s = txt.get_text().strip()
    if not s:
        continue
    # Heuristic: if the text looks numeric or contains parentheses (CI), treat as table numeric column
    if '(' in s or num_regex.match(s):
        txt.set_fontproperties(mono_fp)
# Save a comparison image showing the chosen font while keeping numeric alignment
fig = ax.get_figure()
fig.savefig('font_{}_mono.png'.format(chosen_font.lower()), bbox_inches='tight')
print('Saved', 'font_{}_mono.png'.format(chosen_font.lower()))